# Chunking, Hybrid Retrieval and Semantic Re-Ranking: Practical Solutions for Enhanced LLM Performance in RAG. 


## L1: Retrieval layer

Selecting few candidates out of millions of documents. The goal is pre-select the best 50 documents for a particular query. 

**Constraints**
* Need to consider all the documents (potentially millions)
* Need to run very fast (milliseconds)

**Metric**: Recall@50
* Ratio of relevant document that are present in the top-50. 
Q: 50 is an arbitrary number or is it recommended to retrieve this number ? 
A: So it's a default number. Because this is the number where you we optimize the reranker will run in a certain amount of time. So if you put 100, do we taje it will take a lot more time for the reranker to re-rank because the algorithm is like bigger. So 50 is like arbitrary in that sense. 

#### Option 1: Retrieval with keywords. 

**Idea:**
* **BM25** is used for keyword-based retrieval 
* Each query word is scored using: 
    * Term Frequency (how much a word appear in a document)
    * Inverse Document Frequency (how discriminative is a word across documents)
    * Document length etc 

**Pros:** 
* Scoring is very fast 
* Has been used for decades and still competitive 
* Also works with stemming or lemmatization or word 

**Cons:** 
* Only looks at words present in the query
* Speller / analyzer must be enabled by customer 
* Not semantic 

#### Optin 2: Retrieval with vectors

**Idea:** Encode Query and Documents in an embedding space and use cosine similarity to score. 

**Constraints:** Can perform brute force (KNN) for small index only as computing score for all documents does not scale well. 

**Solution:** Azure AI Search offers Approximate Nearest Neighbours search (HNSW) 

**Pros:** 
* Semantic
* Not based on exact word matching.

**Cons:**
* Performs poorly on short queries and rare words. 

![Descripción de la foto](chunking_strategy.PNG)

We tested the following (using OpenAI ada-002, context = 8192 tokens):

* Impact of using chunks overall: unique chunks vs multiples
* Impact of the chunk size: 512 vs 1024 vs 4096 vs 8192 tokens
* Impact of the text meaning: Preserve sentence boundaries vs hard cut
* Impact of the text position: Chunks with 10% and 25% text overlap vs no overlap 

#### Impact of using chunks overall: unique chunks vs multiples

![Descripción de la foto](SingleVSMultiple.PNG)

#### Impact of the chunk size: 512 vs 1024 vs 4096 vs 8192 tokens

![Descripción de la foto](Chunk_size.PNG)

#### Impact of the text meaning and position: Preserve sentence boundaries vs hard cutand overlap

![Descripción de la foto](Boundaries_Overlapping.PNG)

![Descripción de la foto](Comparing.PNG)

#### Option 3: Hybrid retrieval 

**Idea:** Performs both keyword  and vector retrieval and applies a fusion step to select the best results from each tecchnique

**Azure AI Search** currently uses Reciprocal Rank Fusion (RRF) to produce a single result set

![Descripción de la foto](Comparing2.PNG)

#### Is that enough ? 

Naïve strategy: feed those 50 chunks to the LLM. 
* Pros: No post-processing or re-ranking needed 
* Cons: Potentially a lot of noise, more LLM tokens consumed, more prone to hallucination ? 

Advnaced strategy: re-rank documents to take only the most relevant part to the query
* Pros: Better quality to feed to the LLM leads to better answers
* Cons: add a layer of complexity 

## L2: Re-Ranking layer

**Goal:** re-rank the 50 docs from L1 to have the most relevant first

**Difference with L1**
* Only runs on a small number of documents (50) for each query 
* We can afford model architecture that needs more computing power

**Metric:** NDCG@3
* Normalized measures of how well a model re-ranks a fixed set of documents and put the best in the top-3
* Discounted by rank (top1 is more important than top2 and so on)

![Descripción de la foto](ReRank.PNG)

Comparing the metrics with the others strategies: 

![Descripción de la foto](ReRank2.PNG)

## Other techniques 

There are 2 techniques that are kind of very cool and that you can use to enhance your retrieval layer.

#### Query rewriting

Use a LLM to rewrite a query to express the same thing with different formulations and aggregate results.

![Descripción de la foto](Query_rewrite.PNG)

**Example:**
What is best restaurant in Dublin: 
1. Which restaurant is considered the top dining spot in Dublin ? 
2. Can you recommend the finest eatery in Dublin ?
3. I'm looking for the most highly-rated restaurant in Dublin, any suggestions ? 

#### Extract queries from Documents 

Extract potential queries from documents and use it as an extra field (can be done offline)

![Descripción de la foto](extra.PNG)

## Context + LLM + Prompt 

![Descripción de la foto](RAG.PNG)

#### Metrics for RAG 

Evaluation RAG system is an ongoing topic with many actors and many metrics. There is no consensus yet, most metric also rely on a prompt and an LLM. 

* **Frameworks:** Azure AI studio, RAGAS, TonicAI, DeepEval, LangChain, Haystack, UpTrain, TrueLens, Flash RAG... 
* **Metrics:** Retrieval Score, Groundedness, Relevance, Answer Quality, Safety, Fluency, Coherence, Context Recall, Context PRecision, Context Relevancy, Faithfullness, Answer Relevance,.... 

![Descripción de la foto](Metric.PNG)

Comparision: 

![Descripción de la foto](Proxy.PNG)

## Key learning and insights 

* RAG combines the power of retrieval-based models and generative models to enhance the quality and relevance of generated text
* Long documents need to be chunked into smaller logical pieces to 
    * maximize the potential of vector embedding retrieval approaches
    * allows multiple chunks to be passed to the LLM within its context size limit
* Using Hybrid Retrieval + Semantic Re-Ranking is the most effective approach for improved relevance out-of-the-box in Azure AI Search. 